# Develop the LSTM Model

### Imports

In [38]:
import utility as uf
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Tuple, Dict
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error


### Load dataset

In [39]:
# Load dataset 
data = uf.load_dataset()

### Set bounds and targets

In [40]:
start = "2020-01-01"
val_start = "2024-01-01"
val_end = "2025-01-01"

In [41]:
input_cols = data.columns.drop("Dissolved Oxygen (mg/L)")

In [42]:
target_col = "Dissolved Oxygen (mg/L)"

### Split dataset into three datasets

In [43]:
splits = uf.temporal_splits(data, val_start, val_end)

In [44]:
input_len = 4*24 # Given previous day
output_len = 4*3 # predict the next 3 hours

In [45]:
train_data = uf.SlidingWindowDataset(splits['train'], input_cols, target_col, input_len=input_len, out_len=output_len) # Given previous day predict the next 3 hours
val_data = uf.SlidingWindowDataset(splits['val'], input_cols, target_col, input_len=input_len, out_len=output_len) # Given previous day predict the next 3 hours
test_data = uf.SlidingWindowDataset(splits['test'], input_cols, target_col, input_len=input_len, out_len=output_len) # Given previous day predict the next 3 hours


In [46]:
np.array(train_data[0][0]).shape # 96 points for 36 features

(96, 34)

In [47]:
np.array(train_data[0][1]).shape # 12 points for target feature

(12,)

# LSTM design

In [48]:
class LSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden_size=128, num_layers=2, out_len=24, dropout=0.2):
        super().__init__()
        self.n_features = n_features
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        # map hidden state to output sequence (we'll use the last hidden state)
        self.fc = nn.Linear(hidden_size, out_len)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        output, (h_n, c_n) = self.lstm(x)  # h_n shape: (num_layers, batch, hidden)
        # take last layer hidden state
        last_hidden = h_n[-1]  # (batch, hidden)
        out = self.fc(last_hidden)  # (batch, out_len)
        return out



### LSTM Trainloop

In [ ]:
def train_model_for_windows(
    train_data: Dataset,
    val_data: Dataset,
    test_data: Dataset,
    input_size: int,
    out_len: int,
    batch_size: int = 64,
    n_epochs: int = 50,
    lr: float = 1e-3,
    hidden_size: int = 64,
    num_layers: int = 2,
    device: str = None,
    verbose: bool = True,
    patience: int = 6,
) -> Dict:
    """
    Train an LSTM directly from already-created sliding-window datasets.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, drop_last=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, drop_last=False)

    model = LSTMForecaster(
        n_features=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        out_len=out_len,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    wait = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses)) if train_losses else 0.0

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_losses.append(loss.item())

        val_loss = float(np.mean(val_losses)) if val_losses else 0.0
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if verbose:
            print(f"Epoch {epoch:03d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"Early stopping after {epoch} epochs (patience={patience}). Best val_loss={best_val:.6f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    all_preds = []
    all_truth = []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            preds = model(xb).cpu().numpy()
            all_preds.append(preds)
            all_truth.append(yb.numpy())

    if len(all_preds) == 0:
        test_preds = np.empty((0, out_len), dtype=np.float32)
        test_truth = np.empty((0, out_len), dtype=np.float32)
    else:
        test_preds = np.vstack(all_preds)
        test_truth = np.vstack(all_truth)

    metrics = {
        "mae": float(mean_absolute_error(test_truth, test_preds)),
        "rmse": float(root_mean_squared_error(test_truth, test_preds)),
        "mse": float(mean_squared_error(test_truth, test_preds)),
        "r2": float(r2_score(test_truth, test_preds)),
        "n_samples": int(test_preds.shape[0]),
    }

    return {
        "model": model,
        "history": history,
        "metrics": metrics,
        "test_preds": test_preds,
        "test_truth": test_truth,
        "device": device,
    }


# Train Model

In [52]:
temp_data = train_model_for_windows(
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    input_size=len(input_cols),
    out_len=output_len,
    batch_size=64,
    n_epochs=50,
    lr=1e-3,
    hidden_size=64,
    num_layers=2,
    device=None,
    verbose=True,
    patience=6,
)


Epoch 001 | train_loss=3.557768 | val_loss=1.905484
Epoch 002 | train_loss=0.731985 | val_loss=2.090135
Epoch 003 | train_loss=0.584629 | val_loss=2.011763
Epoch 004 | train_loss=0.511233 | val_loss=2.011498
Epoch 005 | train_loss=0.453657 | val_loss=1.936567
Epoch 006 | train_loss=0.403575 | val_loss=1.640056
Epoch 007 | train_loss=0.377235 | val_loss=1.537925
Epoch 008 | train_loss=0.364267 | val_loss=1.576960
Epoch 009 | train_loss=0.352569 | val_loss=1.579980
Epoch 010 | train_loss=0.341178 | val_loss=1.683855
Epoch 011 | train_loss=0.329923 | val_loss=1.647790
Epoch 012 | train_loss=0.325440 | val_loss=1.664978
Epoch 013 | train_loss=0.312533 | val_loss=1.631064
Early stopping after 13 epochs (patience=6). Best val_loss=1.537925


In [54]:
temp_data['metrics']

{'mae': 0.867102861404419,
 'rmse': 1.224083423614502,
 'mse': 1.5028942823410034,
 'r2': 0.3984409272670746,
 'n_samples': 8653}